## STEP-1 : CREATE CONNECTION WITH SUPABASE DATABASE (POSTGRES)

In [0]:
jdbc_url = "---"

properties = {
    "user": "---",  
    "password": "---",
    "driver": "---"
}

## STEP-2 : CHECK CONNECTION

In [0]:
try:
    df = spark.read.jdbc(
        url=jdbc_url,
        table="students",
        properties=properties
    )
    print("SUCCESS")
    display(df)
except Exception as e:
    print("ERROR:", e)

## STEP-3 : BUILD MENDELIAN DATA ARCHITECTURE FOR STUDENT (SCD)


### READ RAW DATA

In [0]:
students_df = spark.read.jdbc(
    url=jdbc_url,
    table="students",
    properties=properties
)
students_df.count()  

### BRONZE TABLE (RAW LOAD)

In [0]:
students_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("college_dw.bronze.students_raw")

### DISPLAY DATA

In [0]:
display(spark.table("college_dw.bronze.students_raw"))

### CLEAN RAW DATA

In [0]:
from pyspark.sql.functions import col

silver_df = students_df.select(
    col("S_id").alias("student_id"),
    col("S_name").alias("student_name"),
    col("S_mail").alias("student_email"),
    col("S_phone").alias("student_phone"),
    col("S_Address").alias("student_address"),
    col("dep_id").alias("department_id"),
    col("S_admisionYear").alias("admission_year"),
    col("S_dob").alias("date_of_birth")
).dropDuplicates(["student_id"])

### SILVER TABLE (CLEAN DATA)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("college_dw.silver.students_clean")

### CREATE TEMPORARY VIEW

In [0]:
silver_df.createOrReplaceTempView("staging_students")

### GOLD TABLE ( SCD )

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS college_dw.gold.dim_student (
    student_id STRING,
    student_name STRING,
    student_email STRING,
    student_phone STRING,
    student_address STRING,
    department_id STRING,
    admission_year INT,
    date_of_birth DATE,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
)
""")

### HANDEL UPDATES

In [0]:
spark.sql("""
MERGE INTO college_dw.gold.dim_student AS target
USING staging_students AS source
ON target.student_id = source.student_id AND target.is_current = TRUE

WHEN MATCHED AND (
    target.student_name != source.student_name OR
    target.student_email != source.student_email OR
    target.student_phone != source.student_phone OR
    target.student_address != source.student_address OR
    target.department_id != source.department_id OR
    target.admission_year != source.admission_year OR
    target.date_of_birth != source.date_of_birth
)
THEN UPDATE SET
    target.end_date = current_timestamp(),
    target.is_current = FALSE

WHEN NOT MATCHED
THEN INSERT (
    student_id,
    student_name,
    student_email,
    student_phone,
    student_address,
    department_id,
    admission_year,
    date_of_birth,
    start_date,
    end_date,
    is_current
)
VALUES (
    source.student_id,
    source.student_name,
    source.student_email,
    source.student_phone,
    source.student_address,
    source.department_id,
    source.admission_year,
    source.date_of_birth,
    current_timestamp(),
    NULL,
    TRUE
)
""")

### HANDEL DELETION

In [0]:
spark.sql("""
UPDATE college_dw.gold.dim_student
SET 
    end_date = current_timestamp(),
    is_current = FALSE
WHERE is_current = TRUE
AND student_id NOT IN (
    SELECT student_id FROM staging_students
)
""")

### VERIFY

In [0]:
display(spark.table("college_dw.gold.dim_student"))